# Agentic RAG PDF Question-Answering Chatbot

This notebook demonstrates an agentic Retrieval-Augmented Generation (RAG) application. It loads a PDF, converts the document into searchable chunks, creates embeddings, and exposes the retrieval process as a tool that an LLM agent can call.

The example shows how an agent can decide when to use the document retrieval tool and generate an answer based on the retrieved context. The notebook is designed to be easy to follow during portfolio review or self-learning.

In [1]:
# Import the libraries used for PDF loading, chunking, embeddings, and agent creation.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv

# Load environment variables required for API authentication.
load_dotenv()

C:\Users\vishe\AppData\Local\Temp\ipykernel_18316\3035654985.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
# Load the PDF, split it into overlapping chunks, and create an in-memory vector store.
pdf_loader = PyPDFLoader("../data/research_paper.pdf")
loaded_pdf_documents = pdf_loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunked_pdf_documents = text_splitter.split_documents(loaded_pdf_documents)
print("Number of chunks created:", len(chunked_pdf_documents))

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore.from_documents(
    documents=chunked_pdf_documents,
    embedding=embeddings
)

Number of chunks created: 149


In [3]:
# Expose document retrieval as a tool that the agent can call when needed.
from langchain_core.tools import tool


@tool
def retrieve_document_context(user_query: str) -> str:
    """Return the three most relevant passages from the loaded PDF."""
    matching_documents = vector_store.similarity_search(user_query, k=3)
    retrieved_context = ""
    for match in matching_documents:
        retrieved_context += match.page_content
    return retrieved_context

In [4]:
# Initialize the language model that will decide how to use the retrieval tool.
llm = ChatOpenAI(model="gpt-6-luna")

In [5]:
# Create an agent that uses the retrieval tool and follows the response rules.
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[retrieve_document_context],
    system_prompt="""
    Use retrieve_document_context to answer questions from the PDF.
    If you cannot find the answer, say "I don't know".
    """
)

In [6]:
# Invoke the agent with a sample question about the PDF content.
user_query = "What are the components of transfomers and who are the authors?"
response = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]}
)

In [11]:
# Display the assistant's final message from the agent response.
print(response["messages"][-1].content[0]["text"])

The Transformer architecture has two main parts:

- **Encoder:** Each layer contains multi-head self-attention and a feed-forward network. Each sublayer has a residual connection and layer normalization.
- **Decoder:** Each layer contains masked multi-head self-attention, attention over the encoder’s outputs, and a feed-forward network, with residual connections and layer normalization.

The PDF’s authors are **Xavier Amatriain, Ananth Sankar, Jie Bing, Praveen Kumar Bodigutla, Timothy J. Hazen, and Michaeel Kazi**. It says the original Transformer was introduced by **Vaswani et al.** in 2017.
